# 11D — Quarterly decomposition and portfolio evaluation

This notebook evaluates the frozen broad, sector-enriched, and combined critical-minerals signals from Notebook 11C. It asks two separate questions:

1. How much do candidate rankings change because SEC information changes versus because holdings embeddings change?
2. Do the frozen scores predict publication-lagged returns better than chance?

No LLM labels or propagation choices are altered after returns are loaded. Seed firms are excluded from incremental-discovery portfolios.

## 1. Load the frozen handoff before outcomes

The 11C manifest and hashes must validate before this notebook opens the return file.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from IPython.display import display

HERE=Path.cwd().resolve()
if HERE.name=='quarterly_sec_pilot': PILOT_ROOT,GEO_ROOT,ROOT=HERE,HERE.parent,HERE.parents[1]
elif HERE.name=='geopolitical_exposure': GEO_ROOT,ROOT=HERE,HERE.parent;PILOT_ROOT=GEO_ROOT/'quarterly_sec_pilot'
elif HERE.name=='Asset_Embeddings': ROOT=HERE;GEO_ROOT=ROOT/'geopolitical_exposure';PILOT_ROOT=GEO_ROOT/'quarterly_sec_pilot'
else: ROOT=(HERE/'Projects'/'Asset_Embeddings').resolve();GEO_ROOT=ROOT/'geopolitical_exposure';PILOT_ROOT=GEO_ROOT/'quarterly_sec_pilot'
SOURCE=PILOT_ROOT/'three_seed_specifications'; SOURCE_RESULTS=SOURCE/'results'; SOURCE_MANIFESTS=SOURCE/'manifests'
OUT=PILOT_ROOT/'quarterly_portfolio_evaluation'; RESULTS=OUT/'results'; MANIFESTS=OUT/'manifests'
for p in [RESULTS,MANIFESTS]: p.mkdir(parents=True,exist_ok=True)
def sha256(path): return hashlib.sha256(path.read_bytes()).hexdigest()
design=json.loads((SOURCE_MANIFESTS/'three_seed_specification_design.json').read_text())
inventory=pd.read_parquet(SOURCE_MANIFESTS/'output_inventory.parquet')
for row in inventory.itertuples():
    matches=list(SOURCE.rglob(row.artifact)); assert len(matches)==1 and sha256(matches[0])==row.sha256
assert design['outcomes_accessed'] is False
dynamic_scores=pd.read_parquet(SOURCE_RESULTS/'three_specification_propagated_scores.parquet')
seed_panel=pd.read_parquet(SOURCE/'processed'/'automated_quarterly_seed_panel.parquet')
print('Frozen 11C handoff verified:',design['sha256'])
print('Return outcomes loaded yet:',False)

Frozen 11C handoff verified: 75b927535d849fc7072d300971331ced6acb16b900d67f47a990f19fff7e6f46
Return outcomes loaded yet: False


## 2. Construct the frozen text/embedding decomposition

`dynamic` uses text and embeddings from quarter *t*. `fixed_q1_text` holds seed information at Q1 while embeddings change. `fixed_q1_embedding` updates text while holding the Q1 similarity geometry fixed. Analysis is restricted to firms present in all four quarterly embedding universes.

In [2]:
VINTAGES=['2024q1','2024q2','2024q3','2024q4']; MODELS=['rs_binary','rs_l_min']
def load_embedding(vintage,model):
    p=ROOT/'active_passive_extension'/'benchmarks'/'all_s12_control'/vintage/model/'asset_embeddings_10d.parquet'
    f=pd.read_parquet(p).sort_values('permco').reset_index(drop=True); f['permco']=f.permco.astype(int)
    dims=[c for c in f if c.startswith('embedding_')]; a=f[dims].to_numpy(float); n=np.linalg.norm(a,axis=1,keepdims=True)
    return f,np.divide(a,n,out=np.zeros_like(a),where=n>0)
embeddings={(v,m):load_embedding(v,m) for v in VINTAGES for m in MODELS}
common_firms=set.intersection(*[set(f.permco) for f,_ in embeddings.values()])
combined_seed_firms=set(seed_panel.loc[seed_panel.seed_specification.eq('combined'),'firm_id'].astype(int))
evaluation_firms=common_firms-combined_seed_firms

def propagate_to_universe(target_vintage,model,seeds,score_column):
    f,a=embeddings[(target_vintage,model)]; f=f[f.permco.isin(common_firms)].copy().reset_index(drop=True)
    dims=[c for c in f if c.startswith('embedding_')]; x=f[dims].to_numpy(float); n=np.linalg.norm(x,axis=1,keepdims=True); x=np.divide(x,n,out=np.zeros_like(x),where=n>0)
    idx=pd.Series(f.index,index=f.permco); s=seeds[seeds.firm_id.isin(idx.index)].drop_duplicates('firm_id')
    w=s[score_column].to_numpy(float); denom=np.abs(w).sum()
    with np.errstate(over='ignore',invalid='ignore',divide='ignore'):
        score=np.zeros(len(f)) if denom==0 else (x@x[idx.loc[s.firm_id].to_numpy()].T)@w/denom
    if not np.isfinite(score).all(): raise FloatingPointError('Non-finite propagated score encountered')
    return pd.DataFrame({'permco':f.permco,'firm_name':f.firm_name,'propagated_score':score})

rows=[]
for specification in ['broad','sector_enriched','combined']:
  for vintage in VINTAGES:
   for model in MODELS:
    for target in ['relatedness_score','signed_score']:
     current=seed_panel[(seed_panel.seed_specification==specification)&(seed_panel.vintage==vintage)&(seed_panel.channel=='critical_minerals_overall')]
     q1=seed_panel[(seed_panel.seed_specification==specification)&(seed_panel.vintage=='2024q1')&(seed_panel.channel=='critical_minerals_overall')]
     for decomposition,embedding_vintage,seeds in [('dynamic',vintage,current),('fixed_q1_text',vintage,q1),('fixed_q1_embedding','2024q1',current)]:
      z=propagate_to_universe(embedding_vintage,model,seeds,target);z['vintage']=vintage;z['embedding_model']=model
      z['seed_specification']=specification;z['target']=target;z['decomposition']=decomposition;rows.append(z)
decomposition_scores=pd.concat(rows,ignore_index=True)
decomposition_scores['is_any_seed']=decomposition_scores.permco.isin(combined_seed_firms)
decomposition_scores.to_parquet(RESULTS/'quarterly_decomposition_scores.parquet',index=False)
display(pd.DataFrame([{'common_embedding_firms':len(common_firms),'all_seed_firms':len(combined_seed_firms),
                       'common_nonseed_evaluation_firms':len(evaluation_firms),'score_rows':len(decomposition_scores)}]))

,common_embedding_firms,all_seed_firms,common_nonseed_evaluation_firms,score_rows
0,1257,22,1242,181008


## 3. Measure ranking stability and constituent turnover

Top-decile relatedness portfolios and top/bottom-decile signed portfolios are formed on the common non-seed universe. Jaccard overlap and two-way turnover measure constituent stability.

In [3]:
nonseed=decomposition_scores[~decomposition_scores.is_any_seed].copy()
nonseed['percentile']=nonseed.groupby(['vintage','embedding_model','seed_specification','target','decomposition']).propagated_score.rank(pct=True,method='average')
nonseed['portfolio_leg']=np.select([
    (nonseed.target=='relatedness_score')&(nonseed.percentile>=0.9),
    (nonseed.target=='signed_score')&(nonseed.percentile>=0.9),
    (nonseed.target=='signed_score')&(nonseed.percentile<=0.1)],['thematic_long','positive_leg','negative_leg'],default='outside')
members=nonseed[nonseed.portfolio_leg!='outside'].copy()
members.to_parquet(RESULTS/'portfolio_constituents.parquet',index=False)
stability=[]
for keys,g in nonseed.groupby(['embedding_model','seed_specification','target','decomposition']):
 for left,right in zip(VINTAGES[:-1],VINTAGES[1:]):
  a=g[g.vintage==left].set_index('permco').propagated_score;b=g[g.vintage==right].set_index('permco').propagated_score
  joined=pd.concat([a,b],axis=1,join='inner'); joined.columns=['left','right']
  row=dict(zip(['embedding_model','seed_specification','target','decomposition'],keys),from_vintage=left,to_vintage=right,
           firms=len(joined),spearman=joined.left.corr(joined.right,method='spearman'))
  for leg in ['thematic_long','positive_leg','negative_leg']:
   A=set(members[(members.embedding_model==keys[0])&(members.seed_specification==keys[1])&(members.target==keys[2])&(members.decomposition==keys[3])&(members.vintage==left)&(members.portfolio_leg==leg)].permco)
   B=set(members[(members.embedding_model==keys[0])&(members.seed_specification==keys[1])&(members.target==keys[2])&(members.decomposition==keys[3])&(members.vintage==right)&(members.portfolio_leg==leg)].permco)
   if A or B: row[f'{leg}_jaccard']=len(A&B)/len(A|B);row[f'{leg}_turnover']=1-len(A&B)/max(len(A),len(B))
  stability.append(row)
stability=pd.DataFrame(stability);stability.to_parquet(RESULTS/'ranking_and_constituent_stability.parquet',index=False)
display(stability.groupby(['target','decomposition'],as_index=False).agg(
    mean_rank_spearman=('spearman','mean'),mean_thematic_turnover=('thematic_long_turnover','mean'),
    mean_positive_turnover=('positive_leg_turnover','mean'),mean_negative_turnover=('negative_leg_turnover','mean')))

,target,decomposition,mean_rank_spearman,mean_thematic_turnover,mean_positive_turnover,mean_negative_turnover
0,relatedness_score,dynamic,0.804164,0.298222,NaN,NaN
1,relatedness_score,fixed_q1_embedding,0.989991,0.058667,NaN,NaN
2,relatedness_score,fixed_q1_text,0.829786,0.272000,NaN,NaN
3,signed_score,dynamic,0.767364,NaN,0.461333,0.325717
4,signed_score,fixed_q1_embedding,0.961060,NaN,0.201778,0.077509
5,signed_score,fixed_q1_text,0.795193,NaN,0.418667,0.304659


## 4. Attribute quarterly changes to text and embeddings

Dynamic rank changes are compared with each frozen component. Higher agreement with `fixed_q1_text` indicates that changing embedding geometry contributes more; higher agreement with `fixed_q1_embedding` indicates that new text contributes more.

In [4]:
attribution=[]
for keys,g in nonseed.groupby(['vintage','embedding_model','seed_specification','target']):
 w=g.pivot(index='permco',columns='decomposition',values='propagated_score')
 if set(['dynamic','fixed_q1_text','fixed_q1_embedding']).issubset(w):
  attribution.append(dict(zip(['vintage','embedding_model','seed_specification','target'],keys),firms=len(w),
   dynamic_vs_changing_embedding_fixed_text=w.dynamic.corr(w.fixed_q1_text,method='spearman'),
   dynamic_vs_changing_text_fixed_embedding=w.dynamic.corr(w.fixed_q1_embedding,method='spearman')))
attribution=pd.DataFrame(attribution);attribution.to_parquet(RESULTS/'text_embedding_change_attribution.parquet',index=False)
display(attribution.groupby(['target','embedding_model','seed_specification'],as_index=False).agg(
    changing_embedding_agreement=('dynamic_vs_changing_embedding_fixed_text','mean'),
    changing_text_agreement=('dynamic_vs_changing_text_fixed_embedding','mean')))

,target,embedding_model,seed_specification,changing_embedding_agreement,changing_text_agreement
0,relatedness_score,rs_binary,broad,0.944407,0.842247
1,relatedness_score,rs_binary,combined,0.981678,0.888517
2,relatedness_score,rs_binary,sector_enriched,0.997670,0.824887
3,relatedness_score,rs_l_min,broad,0.947046,0.738278
4,relatedness_score,rs_l_min,combined,0.983889,0.861194
5,relatedness_score,rs_l_min,sector_enriched,0.997719,0.850152
6,signed_score,rs_binary,broad,0.964242,0.840968
7,signed_score,rs_binary,combined,0.931790,0.856982
8,signed_score,rs_binary,sector_enriched,0.911894,0.873859
9,signed_score,rs_l_min,broad,0.971105,0.738929


## 5. Open publication-lagged return outcomes

A conservative 60-calendar-day disclosure lag is applied to the S12-based embeddings. The first three complete calendar months after that date define the forward window. Q1 maps to June–August and Q2 to September–November. Q3 has only December locally and is diagnostic; Q4 has no available forward month. Only complete three-month windows enter the primary test.

In [5]:
returns_path=ROOT/'active_passive_extension'/'measures'/'firm_returns_v2_total_return'/'monthly_firm_returns_2024.parquet'
monthly=pd.read_parquet(returns_path);monthly['eom']=pd.to_datetime(monthly.eom);monthly['eom_ret']=pd.to_numeric(monthly.eom_ret,errors='coerce')
formation={'2024q1':'2024-05-30','2024q2':'2024-08-29','2024q3':'2024-11-29','2024q4':'2025-03-01'}
windows={'2024q1':['2024-06-30','2024-07-31','2024-08-31'],
         '2024q2':['2024-09-30','2024-10-31','2024-11-30'],
         '2024q3':['2024-12-31','2025-01-31','2025-02-28'],'2024q4':['2025-04-30','2025-05-31','2025-06-30']}
forward=[]
for vintage,dates in windows.items():
 d=monthly[monthly.eom.isin(pd.to_datetime(dates))].copy()
 f=d.groupby('permco',as_index=False).agg(months=('eom_ret','count'),forward_return=('eom_ret',lambda x:np.prod(1+x.dropna())-1))
 f['vintage']=vintage;f['required_months']=3;f['complete_window']=f.months.eq(3);forward.append(f)
forward=pd.concat(forward,ignore_index=True);forward.to_parquet(RESULTS/'publication_lagged_forward_returns.parquet',index=False)
window_audit=(forward.groupby('vintage',as_index=False).agg(firms=('permco','nunique'),available_months=('months','max'),complete_firms=('complete_window','sum')))
window_audit['formation_date']=window_audit.vintage.map(formation);window_audit.to_parquet(MANIFESTS/'return_window_audit.parquet',index=False)
display(window_audit)
print('Return outcomes are now open. All upstream scores remain frozen.')

,vintage,firms,available_months,complete_firms,formation_date
0,2024q1,4889,3,4752,2024-05-30
1,2024q2,4853,3,4712,2024-08-29
2,2024q3,4761,1,0,2024-11-29


Return outcomes are now open. All upstream scores remain frozen.


## 6. Evaluate cross-sectional ranks and portfolios

The primary sample contains complete forward windows only. Reported statistics are descriptive because just two independent formation quarters are currently available.

In [6]:
evaluation=nonseed.merge(forward,on=['vintage','permco'],how='inner',validate='many_to_one')
primary=evaluation[evaluation.complete_window].copy()
metrics=[]
for keys,g in primary.groupby(['vintage','embedding_model','seed_specification','target','decomposition']):
 top=g[g.percentile>=.9].forward_return;bottom=g[g.percentile<=.1].forward_return
 metrics.append(dict(zip(['vintage','embedding_model','seed_specification','target','decomposition'],keys),
  firms=len(g),rank_correlation=g.propagated_score.corr(g.forward_return,method='spearman'),
  top_decile_return=top.mean(),bottom_decile_return=bottom.mean(),top_minus_bottom=top.mean()-bottom.mean()))
performance=pd.DataFrame(metrics);performance.to_parquet(RESULTS/'primary_forward_performance.parquet',index=False)
display(performance[performance.decomposition.eq('dynamic')].sort_values(
 ['target','embedding_model','seed_specification','vintage']))
display(performance.groupby(['embedding_model','seed_specification','target','decomposition'],as_index=False).agg(
 quarters=('vintage','nunique'),mean_rank_correlation=('rank_correlation','mean'),
 mean_top_decile_return=('top_decile_return','mean'),mean_top_minus_bottom=('top_minus_bottom','mean')))

,vintage,embedding_model,seed_specification,target,decomposition,firms,rank_correlation,top_decile_return,bottom_decile_return,top_minus_bottom
0,2024q1,rs_binary,broad,relatedness_score,dynamic,1242,-0.087910,0.064140,0.091024,-0.026883
36,2024q2,rs_binary,broad,relatedness_score,dynamic,1242,-0.029923,0.101817,0.095231,0.006585
6,2024q1,rs_binary,combined,relatedness_score,dynamic,1242,-0.089193,0.049698,0.104789,-0.055091
42,2024q2,rs_binary,combined,relatedness_score,dynamic,1242,-0.035069,0.070552,0.083222,-0.012669
12,2024q1,rs_binary,sector_enriched,relatedness_score,dynamic,1242,-0.058182,0.043729,0.128471,-0.084742
48,2024q2,rs_binary,sector_enriched,relatedness_score,dynamic,1242,-0.032735,0.075680,0.097116,-0.021436
18,2024q1,rs_l_min,broad,relatedness_score,dynamic,1242,-0.087989,0.047676,0.064011,-0.016334
54,2024q2,rs_l_min,broad,relatedness_score,dynamic,1242,0.000255,0.104154,0.107903,-0.003749
24,2024q1,rs_l_min,combined,relatedness_score,dynamic,1242,-0.043323,0.045820,0.101865,-0.056045
60,2024q2,rs_l_min,combined,relatedness_score,dynamic,1242,-0.038114,0.093265,0.087132,0.006133


,embedding_model,seed_specification,target,decomposition,quarters,mean_rank_correlation,mean_top_decile_return,mean_top_minus_bottom
0,rs_binary,broad,relatedness_score,dynamic,2,-0.058917,0.082979,-0.010149
1,rs_binary,broad,relatedness_score,fixed_q1_embedding,2,-0.051053,0.074034,-0.018263
2,rs_binary,broad,relatedness_score,fixed_q1_text,2,-0.048307,0.080021,-0.003682
3,rs_binary,broad,signed_score,dynamic,2,0.060234,0.107247,0.035719
4,rs_binary,broad,signed_score,fixed_q1_embedding,2,0.057306,0.102575,0.036492
5,rs_binary,broad,signed_score,fixed_q1_text,2,0.059836,0.109774,0.040608
6,rs_binary,combined,relatedness_score,dynamic,2,-0.062131,0.060125,-0.033880
7,rs_binary,combined,relatedness_score,fixed_q1_embedding,2,-0.057387,0.060768,-0.030426
8,rs_binary,combined,relatedness_score,fixed_q1_text,2,-0.057813,0.059391,-0.034822
9,rs_binary,combined,signed_score,dynamic,2,0.069497,0.111720,0.049123


## 7. Matched permutation benchmark

Within each quarter and model, return labels are shuffled across the same firms 500 times. This preserves the score distribution, seed count, portfolio size, and realized cross-section while breaking any score–outcome link.

In [7]:
rng=np.random.default_rng(20240904); placebo=[]; REPS=500
for keys,g in primary.groupby(['vintage','embedding_model','seed_specification','target','decomposition']):
 score=g.propagated_score.to_numpy();ret=g.forward_return.to_numpy();top=g.percentile.to_numpy()>=.9;bottom=g.percentile.to_numpy()<=.1
 actual_corr=pd.Series(score).corr(pd.Series(ret),method='spearman');actual_spread=ret[top].mean()-ret[bottom].mean()
 null_corr=[];null_spread=[]
 for repetition in range(REPS):
  shuffled=rng.permutation(ret);null_corr.append(pd.Series(score).corr(pd.Series(shuffled),method='spearman'));null_spread.append(shuffled[top].mean()-shuffled[bottom].mean())
 placebo.append(dict(zip(['vintage','embedding_model','seed_specification','target','decomposition'],keys),
  actual_rank_correlation=actual_corr,rank_p_two_sided=(1+np.sum(np.abs(null_corr)>=abs(actual_corr)))/(REPS+1),
  actual_top_minus_bottom=actual_spread,spread_p_two_sided=(1+np.sum(np.abs(null_spread)>=abs(actual_spread)))/(REPS+1)))
placebo=pd.DataFrame(placebo);placebo.to_parquet(RESULTS/'matched_permutation_results.parquet',index=False)
display(placebo[placebo.decomposition.eq('dynamic')].sort_values(['target','embedding_model','seed_specification','vintage']))

,vintage,embedding_model,seed_specification,target,decomposition,actual_rank_correlation,rank_p_two_sided,actual_top_minus_bottom,spread_p_two_sided
0,2024q1,rs_binary,broad,relatedness_score,dynamic,-0.087910,0.001996,-0.026883,0.293413
36,2024q2,rs_binary,broad,relatedness_score,dynamic,-0.029923,0.321357,0.006585,0.794411
6,2024q1,rs_binary,combined,relatedness_score,dynamic,-0.089193,0.001996,-0.055091,0.013972
42,2024q2,rs_binary,combined,relatedness_score,dynamic,-0.035069,0.235529,-0.012669,0.670659
12,2024q1,rs_binary,sector_enriched,relatedness_score,dynamic,-0.058182,0.051896,-0.084742,0.001996
48,2024q2,rs_binary,sector_enriched,relatedness_score,dynamic,-0.032735,0.233533,-0.021436,0.415170
18,2024q1,rs_l_min,broad,relatedness_score,dynamic,-0.087989,0.001996,-0.016334,0.457086
54,2024q2,rs_l_min,broad,relatedness_score,dynamic,0.000255,0.988024,-0.003749,0.914172
24,2024q1,rs_l_min,combined,relatedness_score,dynamic,-0.043323,0.139721,-0.056045,0.009980
60,2024q2,rs_l_min,combined,relatedness_score,dynamic,-0.038114,0.221557,0.006133,0.826347


## 8. Freeze the evaluation handoff

The next update can add 2025 returns and a seed-count-matched enriched robustness test without rewriting this version.

In [8]:
artifacts=['quarterly_decomposition_scores.parquet','portfolio_constituents.parquet','ranking_and_constituent_stability.parquet',
 'text_embedding_change_attribution.parquet','publication_lagged_forward_returns.parquet','primary_forward_performance.parquet','matched_permutation_results.parquet']
inv=[]
for name in artifacts:
 p=RESULTS/name;inv.append({'artifact':name,'rows':len(pd.read_parquet(p)),'bytes':p.stat().st_size,'sha256':sha256(p)})
inv=pd.DataFrame(inv);inv.to_parquet(MANIFESTS/'output_inventory.parquet',index=False)
manifest={'stage':'quarterly_portfolio_evaluation_v1','upstream_design_sha256':design['sha256'],
 'return_file_sha256':sha256(returns_path),'disclosure_lag_calendar_days':60,'complete_primary_vintages':['2024q1','2024q2'],
 'incomplete_diagnostic_vintage':'2024q3','unavailable_vintage':'2024q4','permutation_repetitions':REPS,
 'seed_firms_excluded_from_incremental_evaluation':True,'artifacts':inv.to_dict('records')}
manifest['sha256']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()
(MANIFESTS/'evaluation_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
display(inv);print('Evaluation manifest:',manifest['sha256'])

,artifact,rows,bytes,sha256
0,quarterly_decomposition_scores.parquet,181008,1975874,bbc8a238ec8f56f600f5e73cc1c8bbae9f264ece5bd578...
1,portfolio_constituents.parquet,26928,363471,590580d5ec81ee160783347d1ffce8e0ca3632b68b9c9b...
2,ranking_and_constituent_stability.parquet,108,12197,2aaf1e2241982b21f63b8df64a0e605cc4a1116e260274...
3,text_embedding_change_attribution.parquet,48,5803,76a7019f48d28e054f3ebab2403ccac08ee94f8bdd5bb7...
4,publication_lagged_forward_returns.parquet,14503,189460,c7c60d7d984842d7fa709937a40beaf21240c60d1be430...
5,primary_forward_performance.parquet,72,8481,74c6f5b61f28c141f57b6ee598be1038af8e88d2eb2c44...
6,matched_permutation_results.parquet,72,7942,087fd5dbe7b131ece83699cf1265cc0104c2103135b9f9...


Evaluation manifest: fec4b1cd198968635d3912431cfc3df9b5c9fa109435788f45e39574769c511d
